# Dark-Vessel Detection — train on Colab GPU

Trains the center-heatmap detector on a free/cheap Colab GPU using the tile bundle you uploaded to Google Drive.

**Before running:**
1. Menu **Runtime → Change runtime type → T4 GPU** (or better), Save.
2. On your laptop you ran `python scripts/make_colab_bundle.py` and uploaded `outputs/colab_tiles.tar` to a Drive folder called **`darkvessel`**.
3. Then **Runtime → Run all** and approve the Drive permission popup.

Checkpoints are written to `darkvessel/checkpoints/` **on your Drive**, so they survive a Colab disconnect. When training ends (or even mid-run), download `best.pt` from Drive to your laptop at `outputs/checkpoints/best.pt` and continue locally with `scripts/04_predict_scene.py`.

In [1]:
# 1. Confirm a GPU is attached (you should see a Tesla T4 / L4 / A100 table)
!nvidia-smi

Fri Jun 12 15:03:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Get the code
!git clone https://github.com/Harsh-Antares/dark-vessel-detection.git
%cd dark-vessel-detection

Cloning into 'dark-vessel-detection'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 100 (delta 27), reused 80 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 3.02 MiB | 18.95 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/dark-vessel-detection


In [3]:
# 3. Connect Google Drive (a popup asks for permission)
from google.colab import drive
drive.mount('/content/drive')

BUNDLE = '/content/drive/MyDrive/darkvessel/colab_tiles.tar'
CKPT_DIR = '/content/drive/MyDrive/darkvessel/checkpoints'

import os
assert os.path.exists(BUNDLE), f'Bundle not found at {BUNDLE} — upload colab_tiles.tar to the darkvessel folder on Drive.'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Drive connected, bundle found.')

Mounted at /content/drive
Drive connected, bundle found.


In [4]:
# 4. Unpack the tiles onto Colab's fast local disk (~2 minutes)
!mkdir -p outputs
!tar xf {BUNDLE} -C outputs/
!head -2 outputs/tiles/chips_train.csv
!ls outputs/tiles/train | wc -l && ls outputs/tiles/validation | wc -l

chip_path,scene_id,row_off,col_off,n_points,n_vessels
train/00a035722196ee86t_0_8960.npz,00a035722196ee86t,0,8960,0,0
3038
600


In [ ]:
# 5. Train. Checkpoints go straight to Drive.
#    If you hit 'CUDA out of memory', change --batch-size to 4.
!python scripts/03_train.py --epochs 7 --batch-size 8 --checkpoints-dir {CKPT_DIR} --resume-from {CKPT_DIR}/best_29s_e13.pt

Training on device: cuda
Validation capped at 400 random chips (of 600)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 166MB/s]
Resumed weights from /content/drive/MyDrive/darkvessel/checkpoints/best_29s_e5.pt (epoch 4, F1 0.648)
epoch 1:   0% 0/760 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in

## After training

- The best checkpoint (highest validation F1) is at **`Drive → darkvessel → checkpoints → best.pt`**.
- Download it to your laptop and place it at `outputs/checkpoints/best.pt` inside the project.
- Continue locally with full-scene inference:
  ```
  python scripts/04_predict_scene.py --all --split validation
  python scripts/05_dark_split_and_eval.py --split validation
  ```
- If Colab disconnected mid-run: `last.pt` / `best.pt` on Drive are from the most recent completed epochs — often already usable. **Before re-running the notebook, rename them on Drive** (e.g. `best_run1.pt`): a fresh run starts from scratch and will overwrite them with its own early (worse) checkpoints.